# Projet IC1 (1/3) — Du sondage au jeu d’apprentissage

[![Ouvrir dans Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/earthai-tech/lab-courses/blob/main/inphb/ic1/notebooks/IC1_Projet_S3_composites.ipynb)
Prospect aurifère : 40 sondages, échantillons de 1 m.
> Données : dossier `IC1_TP_donnees`. Bibliothèques : pandas, numpy, matplotlib, scikit-learn.

In [ ]:
# Configuration : Colab ou Jupyter local. Exécutez cette cellule en premier.
import os, shutil, urllib.request
DONNEES = ['sondages_miniers.csv']
URL = 'https://raw.githubusercontent.com/earthai-tech/lab-courses/main/inphb/ic1/data/'
for f in DONNEES:
    if os.path.exists(f):
        continue
    if os.path.exists(os.path.join('..', 'data', f)):
        shutil.copy(os.path.join('..', 'data', f), f)   # kit hors ligne / dépôt cloné
    else:
        urllib.request.urlretrieve(URL + f, f)            # Colab : téléchargement
    print('OK', f)


In [ ]:
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split, StratifiedKFold, GroupKFold, cross_val_score, cross_val_predict, learning_curve, GridSearchCV
from sklearn.metrics import mean_absolute_error, r2_score, roc_auc_score, ConfusionMatrixDisplay
m = pd.read_csv('sondages_miniers.csv'); print(m.shape)
m.head()

## 1. QA/QC rapide

In [ ]:
print('de >= a :', (m['de_m'] >= m['a_m']).sum())
print('teneurs <= 0 :', (m['au_gpt'] <= 0).sum())
print(m['lithologie'].value_counts())

## 2. Composites de 2 m
À COMPLÉTER si les échantillons n’ont pas tous 1 m : pondérer par la longueur.

In [ ]:
m['bloc'] = (m['de_m'] // 2).astype(int)
comp = m.groupby(['sondage', 'bloc']).agg(
    x_m=('x_m', 'first'), y_m=('y_m', 'first'), de_m=('de_m', 'min'), a_m=('a_m', 'max'),
    lithologie=('lithologie', lambda s: s.mode()[0]),
    sulfures_pct=('sulfures_pct', 'mean'), alteration=('alteration', 'max'),
    au_gpt=('au_gpt', 'mean')).reset_index().drop(columns='bloc')
comp['mineralise'] = (comp['au_gpt'] > 0.5).astype(int)
print(len(comp), 'composites ;', round(comp['mineralise'].mean(), 3), 'minéralisés')
comp.to_csv('mes_composites.csv', index=False)

## 3. Statistiques et carte

In [ ]:
print(comp.groupby('lithologie')['au_gpt'].describe().round(3))
t = comp.groupby('sondage').agg(x=('x_m', 'first'), y=('y_m', 'first'), au=('au_gpt', 'mean'))
plt.scatter(t.x, t.y, c=np.log10(t.au), cmap='YlOrRd', s=80); plt.colorbar(label='log10 Au moyen'); plt.show()